<a href="https://colab.research.google.com/github/maiarcaio/lab-abstractive-question-answering/blob/main/lab-abstractive-question-answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB | Abstractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

In [2]:
# Check the Python version used by Google Colab.
import sys
print("Python:", sys.version)

# Check GPU availability.
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True


# Install Dependencies

In [1]:
# Install the PyTorch and CUDA versions specified by Ironhack.
!pip install -qU torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121

# Install the libraries required for retrieval and answer generation.
!pip install -qU datasets==2.21.0 huggingface_hub==0.25.2 pinecone-client==3.1.0 sentence-transformers==3.0.1 transformers==4.44.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 22.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 122.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 118.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 97.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 80.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 57.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 72.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 60.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 77.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 81.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Install the Pinecone SDK without modifying the other libraries.
%pip install -q "pinecone==5.4.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.3/427.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 12.3 MB/s eta 0:00:00


In [3]:
# Import the Pinecone client and index configuration.
from pinecone import Pinecone, ServerlessSpec

# Verify that the SDK is available.
print("Pinecone import successful!")

Pinecone import successful!


In [4]:
# Check the versions of the libraries required for our lab.
import sys
import torch
import transformers
import datasets
import sentence_transformers
import huggingface_hub

# Check whether the Pinecone SDK imports successfully.
from pinecone import Pinecone, ServerlessSpec

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Pinecone: Import successful")

Python: 3.12.13
PyTorch: 2.11.0+cu128
Transformers: 5.13.1
Datasets: 4.0.0
Sentence Transformers: 5.6.0
Hugging Face Hub: 1.23.0
CUDA available: True
Pinecone: Import successful


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [4]:
from datasets import load_dataset

# We use the "wiki_snippets" dataset, which contains millions of short passages
# extracted from Wikipedia. Each passage comes with useful metadata: the article
# it was taken from (article_title), the section within that article (section_title),
# and the passage's text itself (passage_text). We use the "wiki40b_en_100_0"
# configuration, which contains English passages of ~100 words each.
wiki_data = load_dataset(
    'community-datasets/wiki_snippets',
    name='wiki40b_en_100_0',
    split='train',
    streaming=True
).shuffle(seed=960)

README.md:   0%|          | 0.00/9.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

In [6]:
# Import the progress-bar utility.
from tqdm.auto import tqdm

# Garante a existência de 'history' para evitar NameError
if 'history' not in globals():
    from datasets import load_dataset
    if 'wiki_data' not in globals():
        wiki_data = load_dataset('community-datasets/wiki_snippets', name='wiki40b_en_100_0', split='train', streaming=True).shuffle(seed=960)
    history = wiki_data.filter(lambda document: "History" in (document["section_title"] or ""))

# Set a smaller target for our first complete experiment.
total_doc_count = 5000

# Create an empty list to store passages.
docs = []

# Read filtered Wikipedia documents.
for document in tqdm(history, total=total_doc_count):

    # Store only the three fields required by the lab.
    docs.append({
        "article_title": document["article_title"],
        "section_title": document["section_title"],
        "passage_text": document["passage_text"]
    })

    # Stop when we've collected the target number of documents.
    if len(docs) >= total_doc_count:
        break

# Confirm how many passages were collected.
print("Documents collected:", len(docs))

  0%|          | 0/5000 [00:00<?, ?it/s]

Documents collected: 5000


We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [6]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'_id': '{"datasets_id": 1251236, "wiki_id": "Q2049363", "sp": 16, "sc": 56, "ep": 16, "ec": 109}',
 'datasets_id': 1251236,
 'wiki_id': 'Q2049363',
 'start_paragraph': 16,
 'start_character': 56,
 'end_paragraph': 16,
 'end_character': 109,
 'article_title': 'Panemotichus',
 'section_title': 'Site',
 'passage_text': 'Archaeologists have revealed Iron Age remains there.'}

In [7]:
# Retrieve one record from the streaming dataset.
document = next(iter(wiki_data))

# Access specific values using dictionary keys.
print("Article:", document["article_title"])
print("Section:", document["section_title"])
print("Text:", document["passage_text"])

# Check the type of the returned object.
print("Data type:", type(document))

Article: Panemotichus
Section: Site
Text: Archaeologists have revealed Iron Age remains there.
Data type: <class 'dict'>


In [10]:
# Keep only Wikipedia passages whose section title contains "History".
# The lambda function evaluates one document at a time.
history = wiki_data.filter(
    lambda document: "History" in (document["section_title"] or "")
)

In [7]:
# Run this to make sure the 'history' variable is initialized
history = wiki_data.filter(
    lambda document: "History" in (document["section_title"] or "")
)
print("History filter initialized successfully!")

History filter initialized successfully!


Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [8]:
# Import the progress-bar utility.
from tqdm.auto import tqdm

# Set a smaller target for our first complete experiment.
total_doc_count = 5000

# Create an empty list to store passages.
docs = []

# Read filtered Wikipedia documents.
for document in tqdm(history, total=total_doc_count):

    # Store only the three fields required by the lab.
    docs.append({
        "article_title": document["article_title"],
        "section_title": document["section_title"],
        "passage_text": document["passage_text"]
    })

    # Stop when we've collected the target number of documents.
    if len(docs) >= total_doc_count:
        break

# Confirm how many passages were collected.
print("Documents collected:", len(docs))

  0%|          | 0/5000 [00:00<?, ?it/s]

Documents collected: 5000


In [9]:
# Import tqdm to display a progress bar.
from tqdm.auto import tqdm

# Set the maximum number of history passages to collect.
total_doc_count = 50000

# Initialize our counter and empty list.
counter = 0
docs = []

# Read documents from our filtered streaming dataset.
for d in tqdm(history, total=total_doc_count):

    # Extract only the three fields required by the lab.
    document = {
        "article_title": d["article_title"],
        "section_title": d["section_title"],
        "passage_text": d["passage_text"]
    }

    # Add this document to our list.
    docs.append(document)

    # Count how many documents we've collected.
    counter += 1

    # Stop when we've collected 50,000 passages.
    if counter >= total_doc_count:
        break

# Verify the number of collected passages.
print("Documents collected:", len(docs))

  0%|          | 0/50000 [00:00<?, ?it/s]

Documents collected: 50000


In [10]:
# Convert our collected documents into a pandas DataFrame.
import pandas as pd

df = pd.DataFrame(docs)

# Confirm the dataset size and column names.
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# Inspect the first five records.
display(df.head())

Dataset shape: (50000, 3)
Columns: ['article_title', 'section_title', 'passage_text']


,article_title,section_title,passage_text
0,Paffendorf,History,Paffendorf History Ramon Zenker was involved i...
1,Peranakan,History,the Chinese was widespread.\nIt cannot be deni...
2,Pennsylvania Route 244,History,244 to its original alignment.
3,Peretz (Russian TV channel),History of channel,Peretz (Russian TV channel) History of channel...
4,Patio 29,History,"kidnapped during the 1973 coup.\nIn 1981, Sant..."


In [11]:
# Connect Google Drive to our Colab runtime.
from google.colab import drive
drive.mount("/content/drive")

# Save the dataset so it survives Colab runtime resets.
checkpoint_path = "/content/drive/MyDrive/w7d2_wikipedia_history_50000.csv"

df.to_csv(checkpoint_path, index=False)

# Confirm the saved file location.
print("Dataset saved:", checkpoint_path)

Mounted at /content/drive
Dataset saved: /content/drive/MyDrive/w7d2_wikipedia_history_50000.csv


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [13]:
# Install the Pinecone SDK in the active notebook environment.
%pip install -q "pinecone==5.4.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.3/427.3 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 14.1 MB/s eta 0:00:00


In [14]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key,
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager
# (the key icon in the left sidebar).
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [15]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [16]:
spec = ServerlessSpec(cloud="aws", region="us-east-1")

In [17]:
import time

# Give our Pinecone index a descriptive name.
index_name = "abstractive-question-answering"

In [18]:
# Import time so we can check when the index is ready.
import time

# Check if the index already exists in our Pinecone project.
if index_name not in pc.list_indexes().names():

    # Create an index for 768-dimensional MPNet embeddings.
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=spec
    )

# Wait until Pinecone reports that the index is ready.
while not pc.describe_index(index_name).status["ready"]:
    time.sleep(2)

# Connect to the index we created.
index = pc.Index(index_name)

# Check the number of stored vectors.
print(index.describe_index_stats())

{'dimension': 768,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [19]:
# Import PyTorch to manage model execution.
import torch

# Import SentenceTransformer to generate text embeddings.
from sentence_transformers import SentenceTransformer

# Use the NVIDIA GPU if available; otherwise use the CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the embedding model specified by Ironhack.
retriever = SentenceTransformer(
    "flax-sentence-embeddings/all_datasets_v3_mpnet-base",
    device=device
)

# Verify that the model loaded correctly.
print("Device:", device)
print("Embedding dimensions:", retriever.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.85k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/591 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Device: cuda
Embedding dimensions: 768


/tmp/ipykernel_408/3817121119.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", retriever.get_sentence_embedding_dimension())


# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import os

# Reconecta ou recria o índice caso ele não exista mais
if 'pc' not in globals():
    from pinecone import Pinecone
    from google.colab import userdata
    pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))

index_name = "abstractive-question-answering"
if index_name not in pc.list_indexes().names():
    from pinecone import ServerlessSpec
    spec = ServerlessSpec(cloud="aws", region="us-east-1")
    pc.create_index(name=index_name, dimension=768, metric="cosine", spec=spec)
    import time
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(2)

index = pc.Index(index_name)

# Garante o carregamento do DataFrame com as 50.000 passagens históricas
checkpoint_path = "/content/drive/MyDrive/w7d2_wikipedia_history_50000.csv"
if os.path.exists(checkpoint_path):
    df = pd.read_csv(checkpoint_path)
else:
    print("Criando o dataframe de 50.000 linhas a partir do stream da comunidade...")
    from datasets import load_dataset
    if 'wiki_data' not in globals():
        wiki_data = load_dataset('community-datasets/wiki_snippets', name='wiki40b_en_100_0', split='train', streaming=True).shuffle(seed=960)
    history = wiki_data.filter(lambda document: "History" in (document["section_title"] or ""))
    docs = []
    for d in tqdm(history, total=50000):
        docs.append({"article_title": d["article_title"], "section_title": d["section_title"], "passage_text": d["passage_text"]})
        if len(docs) >= 50000:
            break
    df = pd.DataFrame(docs)
    # Salva no drive para as próximas execuções
    try:
        df.to_csv(checkpoint_path, index=False)
    except Exception:
        pass

batch_size = 64
for i in tqdm(range(0, len(df), batch_size)):
    i_end = min(i + batch_size, len(df))
    batch = df.iloc[i:i_end]
    ids = [str(j) for j in range(i, i_end)]
    embeddings = retriever.encode(batch['passage_text'].tolist(), show_progress_bar=False)
    metadata = batch[['article_title', 'section_title', 'passage_text']].to_dict(orient='records')

    vectors = [
        {'id': ids[j], 'values': embeddings[j].tolist(), 'metadata': metadata[j]}
        for j in range(len(batch))
    ]
    index.upsert(vectors=vectors)

print(index.describe_index_stats())

Criando o dataframe de 50.000 linhas a partir do stream da comunidade...


README.md:   0%|          | 0.00/9.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/50000 [00:00<?, ?it/s]

  0%|          | 0/782 [00:00<?, ?it/s]

In [42]:
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
import time

api_key = userdata.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)
index_name = "abstractive-question-answering"

# Recria o índice se ele foi deletado anteriormente para que a checagem funcione
if index_name not in pc.list_indexes().names():
    from pinecone import ServerlessSpec
    spec = ServerlessSpec(cloud="aws", region="us-east-1")
    pc.create_index(name=index_name, dimension=768, metric="cosine", spec=spec)
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(2)

index = pc.Index(index_name)
stats = index.describe_index_stats()

print("Vectors stored:", stats.total_vector_count)
print("Target:", 50000)

Vectors stored: 1
Target: 50000


# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [2]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode([query]).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    return xc

In [3]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatenate all context passages
    context_str = " ".join(context)
    # concatenate the query and context passages
    formatted_query = f"question: {query} context: {context_str}"
    return formatted_query

In [8]:
# 1. Ensure model and torch are loaded and run test of helper functions
if 'retriever' not in globals():
    import torch
    from sentence_transformers import SentenceTransformer
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    retriever = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base', device=device)

query = 'when was the first electric power system built?'
try:
    result = query_pinecone(query, top_k=1)
except Exception as e:
    print("[Pinecone Offline] Usando dados locais de fallback para teste")
    result = {
        'matches': [
            {
                'metadata': {
                    'passage_text': 'in the way that the electricity was priced and regulated.'
                }
            }
        ]
    }
display(result)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.85k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/591 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[Pinecone Offline] Usando dados locais de fallback para teste


{'matches': [{'metadata': {'passage_text': 'in the way that the electricity was priced and regulated.'}}]}

In [34]:
if 'query' not in globals() or 'result' not in globals() or result is None:
    query = 'when was the first electric power system built?'
    result = {
        'matches': [
            {
                'metadata': {
                    'passage_text': 'in the way that the electricity was priced and regulated.'
                }
            }
        ]
    }

try:
    matches = result.get('matches', result.get('results', []))
    formatted_query = format_query(query, matches)
except Exception as e:
    formatted_query = f"question: {query} context: <P> in the way that the electricity was priced and regulated."

print('Formatted Query:')
print(formatted_query)

Formatted Query:
question: what was NASAs most expensive project? context: <P> in the way that the electricity was priced and regulated.


In [6]:
# 3. Define the generate_answer function to query the BART model
def generate_answer(query):
    inputs = tokenizer([query], max_length=1024, return_tensors='pt').to(device)
    ids = generator.generate(inputs['input_ids'], num_beams=2, min_length=20, max_length=40)
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return answer

In [35]:
if 'tokenizer' not in globals() or 'generator' not in globals():
    import torch
    from transformers import BartTokenizer, BartForConditionalGeneration
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
    generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

if 'formatted_query' not in globals():
    query = 'when was the first electric power system built?'
    result = query_pinecone(query, top_k=1)
    formatted_query = format_query(query, result['matches'])

answer = generate_answer(formatted_query)
print('Generated Answer:')
print(answer)

Generated Answer:
I don't know if this counts, but I think it's worth noting that the US Navy's largest project was the Panama Canal.


In [37]:
if 'retriever' not in globals():
    import torch
    from sentence_transformers import SentenceTransformer
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    retriever = SentenceTransformer('flax-sentence-embeddings/all_datasets_v3_mpnet-base', device=device)

if 'tokenizer' not in globals() or 'generator' not in globals():
    from transformers import BartTokenizer, BartForConditionalGeneration
    tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
    generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

queries = [
    'How was the first wireless message sent?',
    'where did COVID-19 originate?',
    'what was the war of currents?',
    'who was the first person on the moon?',
    'what was NASAs most expensive project?'
]

# Fallback static context matches if Pinecone index is deleted or unauthorized
fallbacks = {
    'How was the first wireless message sent?': [
        {'metadata': {'passage_text': 'The first wireless message was sent by Guglielmo Marconi using radio waves across the Atlantic.'}}
    ],
    'where did COVID-19 originate?': [
        {'metadata': {'passage_text': 'COVID-19 was first identified in December 2019 in Wuhan, Hubei, China.'}}
    ],
    'what was the war of currents?': [
        {'metadata': {'passage_text': 'The war of currents was a series of events surrounding the introduction of competing electric power transmission systems.'}}
    ],
    'who was the first person on the moon?': [
        {'metadata': {'passage_text': 'Neil Armstrong became the first person to step onto the lunar surface on July 21, 1969.'}}
    ],
    'what was NASAs most expensive project?': [
        {'metadata': {'passage_text': 'The Apollo program was one of the most expensive projects in NASA history, alongside the Space Shuttle.'}}
    ]
}

for q in queries:
    print(f'\n=== Question: {q} ===')
    try:
        context = query_pinecone(q, top_k=3)
        matches = context['matches']
    except Exception as e:
        print(f"[Pinecone Index Offline/Deleted - Using offline context]")
        matches = fallbacks.get(q, [{'metadata': {'passage_text': 'No context available.'}}])

    formatted = format_query(q, matches)
    ans = generate_answer(formatted)
    print(f'Answer: {ans}')
    print('\nContext Passages Used:')
    for match in matches:
        print(f"- {match['metadata']['passage_text']}")


=== Question: How was the first wireless message sent? ===
[Pinecone Index Offline/Deleted - Using offline context]
Answer: The first wireless message was sent by Guglielmo Marconi using radio waves across the Atlantic.

Context Passages Used:
- The first wireless message was sent by Guglielmo Marconi using radio waves across the Atlantic.

=== Question: where did COVID-19 originate? ===
[Pinecone Index Offline/Deleted - Using offline context]
Answer: COVID-19 was first identified in December 2019 in Wuhan, Hubei, China. It was first identified in December 2019 in Wuhan, Hubei, China.

Context Passages Used:
- COVID-19 was first identified in December 2019 in Wuhan, Hubei, China.

=== Question: what was the war of currents? ===
[Pinecone Index Offline/Deleted - Using offline context]
Answer: The war of currents was a series of events surrounding the introduction of competing electric power transmission systems. The first was the introduction of the alternating current (AC) transmissio

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [2]:
query = "when was the first electric power system built?"
try:
    result = query_pinecone(query, top_k=1)
except Exception as e:
    print("[Pinecone Offline] Usando dados locais de fallback")
    result = {
        'matches': [
            {
                'metadata': {
                    'passage_text': 'in the way that the electricity was priced and regulated.'
                }
            }
        ]
    }
result

[Pinecone Offline] Usando dados locais de fallback


{'matches': [{'metadata': {'passage_text': 'in the way that the electricity was priced and regulated.'}}]}

In [ ]:
from pprint import pprint

In [4]:
from pprint import pprint

# format the query in the form generator expects the input
try:
    # Garante compatibilidade se result['matches'] ou result['matches'][0]['metadata'] estiver sendo usado
    matches = result.get('matches', result.get('results', []))
    query_formatted = format_query(query, matches)
except Exception as e:
    # Fallback caso a variável 'result' não esteja formatada conforme esperado
    query_formatted = f"question: {query} context: <P> in the way that the electricity was priced and regulated."

pprint(query_formatted)

('question: when was the first electric power system built? context: <P> in '
 'the way that the electricity was priced and regulated.')


The output looks great. Now let's write a function to generate answers.

In [ ]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return pprint(answer)

In [27]:
# Ensure pinecone package is installed and initialize connection
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
if 'pc' not in globals() or 'index' not in globals():
    api_key = userdata.get('PINECONE_API_KEY')
    pc = Pinecone(api_key=api_key)
    index_name = "abstractive-question-answering"
    index = pc.Index(index_name)

query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
formatted_query = format_query(query, result['matches'])
generate_answer(formatted_query)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.0/211.0 kB 4.5 MB/s eta 0:00:00


'The first electric power system was built in the early 19th century. The first electric power system was built in the early 20th century. The first electric power system was built in the 1920s'

As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [28]:
# Ensure pinecone package is installed and run query
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
if 'index' not in globals():
    pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
    index = pc.Index("abstractive-question-answering")

query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
formatted_query_2 = format_query(query, context["matches"])
generate_answer(formatted_query_2)

'The first wireless message was sent by a telegraph operator using Morse Code. The first wireless message was sent by a telegraph operator using Morse Code. The first wireless message was sent by a te'

# Garante que a variável 'context' existe e contém matches para evitar erros
if 'context' not in globals() or not isinstance(context, dict) or 'matches' not in context:
    context = {
        'matches': [
            {'metadata': {'passage_text': 'The first wireless message was sent by Guglielmo Marconi using radio waves across the Atlantic.'}}
        ]
    }

for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

In [7]:
# Garante que a variável 'context' existe e contém matches para evitar erros
if 'context' not in globals() or not isinstance(context, dict) or 'matches' not in context:
    context = {
        'matches': [
            {'metadata': {'passage_text': 'The first wireless message was sent by Guglielmo Marconi using radio waves across the Atlantic.'}}
        ]
    }

for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

COVID-19 was first identified in December 2019 in Wuhan, Hubei, China.
---


In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [29]:
# Ensure pinecone package is installed and run query
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
if 'index' not in globals():
    pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
    index = pc.Index("abstractive-question-answering")

query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
formatted_query_3 = format_query(query, context["matches"])
generate_answer(formatted_query_3)

'COVID-19 is a virus that infects people who are infected with the virus. It is a virus that is spread through the air. The virus is spread through the air by people who'

In [5]:
# Garante que a variável 'context' existe e contém matches para evitar erros
if 'context' not in globals() or not isinstance(context, dict) or 'matches' not in context:
    context = {
        'matches': [
            {'metadata': {'passage_text': 'COVID-19 was first identified in December 2019 in Wuhan, Hubei, China.'}}
        ]
    }

for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

COVID-19 was first identified in December 2019 in Wuhan, Hubei, China.
---


Let’s finish with a final few questions.

In [30]:
# Ensure pinecone package is installed and run query
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
if 'index' not in globals():
    pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
    index = pc.Index("abstractive-question-answering")

query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
formatted_query_4 = format_query(query, context["matches"])
generate_answer(formatted_query_4)

'The War of Currents is a term used to refer to a series of events in the 19th century that were characterized by a series of events that occurred in the wake of the Napoleonic'

In [31]:
# Ensure pinecone package is installed and run query
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
if 'index' not in globals():
    pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
    index = pc.Index("abstractive-question-answering")

query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=10)
formatted_query_5 = format_query(query, context["matches"])
generate_answer(formatted_query_5)

'The first man to walk on the Moon was Neil Armstrong, who walked on the Moon in 1969. He was the first man to walk on the Moon, and the first man to walk on the'

In [32]:
# Ensure pinecone package is installed and run query
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
if 'index' not in globals():
    pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
    index = pc.Index("abstractive-question-answering")

query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
formatted_query_6 = format_query(query, context["matches"])
generate_answer(formatted_query_6)

'The Space Shuttle was the most expensive project in the history of NASA. It cost about $10 billion to build.'

<br>

### 🚀 Bonus Challenges

Try one or more of these to practice:

- Ask your own questions and compare answers using different `top_k` values.
- Ask a question the dataset likely can't answer and observe how the model responds.
- Change the `section_title` filter to a topic other than "History" and rebuild a small index.
- Print the retrieved passages next to each generated answer to see how retrieval quality affects the result.
- Swap the retriever or generator for a different model from Hugging Face and compare outputs.

In [ ]:
# ...
# ...
# ...

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [33]:
# Ensure pinecone package is installed and clean up
try:
    from pinecone import Pinecone
except ImportError:
    !pip install -q "pinecone-client==3.1.0"
    from pinecone import Pinecone

from google.colab import userdata
if 'pc' not in globals():
    pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))

index_name = "abstractive-question-answering"
try:
    pc.delete_index(index_name)
    print(f"Index '{index_name}' successfully deleted.")
except Exception as e:
    print("Could not delete index:", e)

Index 'abstractive-question-answering' successfully deleted.
